# TrackEasy Fraud Detection - ML Training Notebook
## Trains 3 Kaggle datasets with SMOTE balancing -> exports fraud_model.pkl

**Datasets required - download from Kaggle, place in `./datasets/`:**
- `datasets/ds1_fraud.csv` - Kevin Vagan fraud-detection-dataset
- `datasets/ds2_ecommerce.csv` - Smayanj e-commerce-transactions-dataset
- `datasets/ds3_fraudulent.csv` - Shriyash Jagtap fraudulent-e-commerce-transactions

**Output files generated:**
- `model/fraud_model.pkl` - XGBoost classifier (primary)
- `model/fraud_model_lgb.pkl` - LightGBM (backup)
- `model/scaler.pkl` - StandardScaler
- `model/label_encoders.pkl` - Categorical encoders
- `model/feature_columns.json` - Exact column list for inference
- `model/model_meta.json` - Threshold, metrics, config
- `ml_server.py` - Flask inference server (port 5003)

## 1. Install dependencies

In [1]:
import subprocess, sys

packages = [
    "pandas", "numpy", "scikit-learn", "xgboost",
    "imbalanced-learn", "matplotlib", "seaborn",
    "joblib", "lightgbm", "flask"
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


All packages installed



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 2. Imports and configuration

In [4]:
import pandas as pd
import numpy as np
import json, os, warnings, joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_recall_curve,
    average_precision_score, ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
TEST_SIZE    = 0.20
SMOTE_K      = 5

os.makedirs("model", exist_ok=True)
os.makedirs("datasets", exist_ok=True)

print(f"Train : {int((1-TEST_SIZE)*100)}%  |  Test : {int(TEST_SIZE*100)}%  |  SMOTE k={SMOTE_K}")

Train : 80%  |  Test : 20%  |  SMOTE k=5


## 3. Column mapping configuration
Edit these dicts if your CSV column names differ.
Internal names map directly to fraudServer.js rule logic.

In [5]:
# Map actual_csv_column_name -> internal_name

DS1_COL_MAP = {
    'transaction_amount'  : 'transaction_amount',
    'payment_method'      : 'payment_method',
    'device_type'         : 'device_type',
    'ip_address'          : 'ip_address',
    'account_age'         : 'account_age_days',
    'failed_attempts'     : 'failed_payment_count',
    'transaction_hour'    : 'transaction_hour',
    'quantity'            : 'quantity',
    'is_fraud'            : 'label',
}

DS2_COL_MAP = {
    'TransactionAmount'   : 'transaction_amount',
    'PaymentMethod'       : 'payment_method',
    'DeviceType'          : 'device_type',
    'IPAddress'           : 'ip_address',
    'AccountAgeDays'      : 'account_age_days',
    'FailedAttempts'      : 'failed_payment_count',
    'TransactionHour'     : 'transaction_hour',
    'Quantity'            : 'quantity',
    'ProductCategory'     : 'product_category',
    'FraudLabel'          : 'label',
}

DS3_COL_MAP = {
    'Transaction Amount'  : 'transaction_amount',
    'Payment Method'      : 'payment_method',
    'Device Used'         : 'device_type',
    'IP Address'          : 'ip_address',
    'Account Age Days'    : 'account_age_days',
    'Transaction Hour'    : 'transaction_hour',
    'Quantity'            : 'quantity',
    'Customer Age'        : 'customer_age',
    'Product Category'    : 'product_category',
    'Is Fraudulent'       : 'label',
    'Shipping Address'    : '_shipping',
    'Billing Address'     : '_billing',
}

print("Column maps ready")
print("DS1 label col:", [v for k,v in DS1_COL_MAP.items() if v=='label'])
print("DS2 label col:", [v for k,v in DS2_COL_MAP.items() if v=='label'])
print("DS3 label col:", [v for k,v in DS3_COL_MAP.items() if v=='label'])

Column maps ready
DS1 label col: ['label']
DS2 label col: ['label']
DS3 label col: ['label']


## 4. Dataset loader and normaliser

In [6]:
def load_and_normalise(path, col_map, name):
    print(f"\n{'='*52}")
    print(f"  Loading {name}")
    print(f"{'='*52}")

    if not os.path.exists(path):
        print(f"  FILE NOT FOUND - skipping")
        print(f"  Place the CSV at: {path}")
        return None

    df = pd.read_csv(path)
    print(f"  Raw shape : {df.shape}")
    print(f"  Raw cols  : {list(df.columns)[:8]}{'...' if len(df.columns)>8 else ''}")

    # Rename to internal names
    rename = {k: v for k, v in col_map.items() if k in df.columns}
    df = df.rename(columns=rename)
    print(f"  Renamed   : {len(rename)} columns")

    # Derive addresses_match flag (Dataset 3)
    if '_shipping' in df.columns and '_billing' in df.columns:
        df['addresses_match'] = (df['_shipping'] == df['_billing']).astype(int)
        df.drop(columns=['_shipping', '_billing'], inplace=True)
        print("  Derived addresses_match")
    else:
        df['addresses_match'] = 1

    # Extract transaction_hour from datetime columns if missing
    if 'transaction_hour' not in df.columns:
        for col in df.columns:
            if 'date' in col.lower() or 'timestamp' in col.lower():
                try:
                    parsed = pd.to_datetime(df[col], errors='coerce')
                    df['transaction_hour'] = parsed.dt.hour
                    print(f"  Extracted transaction_hour from '{col}'")
                    break
                except Exception:
                    pass

    # Fill missing columns with sensible defaults
    defaults = {
        'account_age_days': 365, 'failed_payment_count': 0,
        'quantity': 1, 'customer_age': 30,
        'product_category': 'unknown', 'transaction_hour': 12,
    }
    for col, val in defaults.items():
        if col not in df.columns:
            df[col] = val
            print(f"  Default {col} = {val}")

    # Validate label
    if 'label' not in df.columns:
        print(f"  ERROR: No label column! Available: {list(df.columns)}")
        return None

    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)

    # Drop IP (geo rules use it, not ML features)
    if 'ip_address' in df.columns:
        df.drop(columns=['ip_address'], inplace=True)

    # Drop ID / PII
    drop_pats = ['id', 'name', 'email', 'address', 'phone']
    to_drop = [c for c in df.columns
               if any(p in c.lower() for p in drop_pats)
               and c not in ('addresses_match',)]
    if to_drop:
        df.drop(columns=to_drop, inplace=True)
        print(f"  Dropped PII: {to_drop}")

    vc = df['label'].value_counts()
    fraud_rate = vc.get(1,0) / len(df) * 100
    print(f"  Clean: {df.shape}  |  Legit={vc.get(0,0):,}  Fraud={vc.get(1,0):,} ({fraud_rate:.1f}%)")
    return df

# Load all 3 datasets
df1 = load_and_normalise("datasets/ds1_fraud.csv",      DS1_COL_MAP, "Dataset 1 - Kevin Vagan")
df2 = load_and_normalise("datasets/ds2_ecommerce.csv",  DS2_COL_MAP, "Dataset 2 - Smayanj")
df3 = load_and_normalise("datasets/ds3_fraudulent.csv", DS3_COL_MAP, "Dataset 3 - Shriyash Jagtap")

loaded = [d for d in [df1, df2, df3] if d is not None]
print(f"\n{len(loaded)}/3 datasets loaded")


  Loading Dataset 1 - Kevin Vagan
  FILE NOT FOUND - skipping
  Place the CSV at: datasets/ds1_fraud.csv

  Loading Dataset 2 - Smayanj
  FILE NOT FOUND - skipping
  Place the CSV at: datasets/ds2_ecommerce.csv

  Loading Dataset 3 - Shriyash Jagtap
  FILE NOT FOUND - skipping
  Place the CSV at: datasets/ds3_fraudulent.csv

0/3 datasets loaded


## 5. Merge all datasets

In [7]:
if not loaded:
    raise RuntimeError("No datasets loaded. Place CSV files in ./datasets/ and re-run cell 4.")

df_all = pd.concat(loaded, ignore_index=True)
print(f"Combined shape : {df_all.shape}")
print(f"All columns    : {list(df_all.columns)}")

# Fill cross-dataset NaNs
num_cols = df_all.select_dtypes(include=[np.number]).columns
df_all[num_cols] = df_all[num_cols].fillna(df_all[num_cols].median())
cat_cols_raw = df_all.select_dtypes(include=['object']).columns
for c in cat_cols_raw:
    df_all[c] = df_all[c].fillna('unknown')

fraud_n = df_all['label'].sum()
legit_n = (df_all['label'] == 0).sum()
print(f"\nLegit (0) : {legit_n:,}")
print(f"Fraud (1) : {fraud_n:,}")
print(f"Ratio     : 1 fraud per {legit_n // max(fraud_n,1):.0f} legit")

df_all.head()

RuntimeError: No datasets loaded. Place CSV files in ./datasets/ and re-run cell 4.

## 6. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

cols_to_plot = [
    'transaction_amount','account_age_days','quantity',
    'failed_payment_count','transaction_hour','addresses_match'
]

for ax, col in zip(axes, cols_to_plot):
    if col not in df_all.columns:
        ax.text(0.5,0.5,f"{col}\nnot in dataset",ha='center',va='center',
                transform=ax.transAxes,color='gray')
        ax.set_title(col)
        continue

    fraud_v = df_all[df_all['label']==1][col]
    legit_v = df_all[df_all['label']==0][col]

    if col == 'addresses_match':
        counts = df_all.groupby(['addresses_match','label']).size().unstack(fill_value=0)
        counts.plot(kind='bar', ax=ax, color=['#4a90d9','#e74c3c'], width=0.6, legend=True)
        ax.set_xlabel('Addresses Match (0=no, 1=yes)')
    else:
        ax.hist(legit_v.clip(upper=legit_v.quantile(0.99)),
                bins=40, alpha=0.6, color='#4a90d9', label='Legit', density=True)
        ax.hist(fraud_v.clip(upper=fraud_v.quantile(0.99)),
                bins=40, alpha=0.6, color='#e74c3c', label='Fraud', density=True)
        ax.legend(fontsize=8)

    ax.set_title(col, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

plt.suptitle("Feature Distributions: Fraud vs Legit", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("model/eda_distributions.png", dpi=120, bbox_inches='tight')
plt.show()

## 7. Encode categorical features

In [ ]:
CATEGORICAL_COLS = ['payment_method', 'device_type', 'product_category']
NUMERIC_COLS    = [
    'transaction_amount', 'account_age_days', 'failed_payment_count',
    'transaction_hour', 'quantity', 'addresses_match', 'customer_age'
]

label_encoders = {}

for col in CATEGORICAL_COLS:
    if col in df_all.columns:
        le = LabelEncoder()
        df_all[col] = le.fit_transform(df_all[col].astype(str).str.lower().str.strip())
        label_encoders[col] = le
        print(f"  '{col}' classes: {list(le.classes_)}")

FEATURE_COLS = [c for c in (NUMERIC_COLS + CATEGORICAL_COLS) if c in df_all.columns]

print(f"\nFinal {len(FEATURE_COLS)} features:")
rule_map = {
    'quantity'            : 'Rule 4 - 3x avg anomaly (+7)',
    'failed_payment_count': 'Rule 2 - payment failures (+4)',
    'transaction_hour'    : 'Rule 3 - speed check (+3)',
    'account_age_days'    : 'Rule 4 amplifier - new account',
    'transaction_amount'  : 'Rule 4 amplifier - high value',
    'addresses_match'     : 'Rule 5 support - billing mismatch',
    'payment_method'      : 'Order model - payment anomaly',
    'device_type'         : 'UserSession - device switching',
    'customer_age'        : 'User model - demographic signal',
    'product_category'    : 'Order model - category risk',
}
for i, c in enumerate(FEATURE_COLS, 1):
    print(f"  {i:2}. {c:<26} {rule_map.get(c,'general signal')}")

with open("model/feature_columns.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
print("\nSaved model/feature_columns.json")

## 8. Train / Test split (80/20, stratified)

In [ ]:
X = df_all[FEATURE_COLS].values
y = df_all['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Train : {X_train.shape[0]:,} rows  Fraud={y_train.sum():,} ({y_train.mean()*100:.1f}%)")
print(f"Test  : {X_test.shape[0]:,}  rows  Fraud={y_test.sum():,} ({y_test.mean()*100:.1f}%)")
print(f"Feats : {X_train.shape[1]}")

## 9. Scale numeric features (fit on train only)

In [ ]:
numeric_indices = [i for i,c in enumerate(FEATURE_COLS) if c in NUMERIC_COLS]
cat_indices     = [i for i,c in enumerate(FEATURE_COLS) if c in CATEGORICAL_COLS]

print(f"Scaling columns: {[FEATURE_COLS[i] for i in numeric_indices]}")

scaler = StandardScaler()
X_train_scaled = X_train.astype(float).copy()
X_test_scaled  = X_test.astype(float).copy()

if numeric_indices:
    X_train_scaled[:, numeric_indices] = scaler.fit_transform(X_train[:, numeric_indices])
    X_test_scaled[:, numeric_indices]  = scaler.transform(X_test[:, numeric_indices])

joblib.dump(scaler, "model/scaler.pkl")
joblib.dump(label_encoders, "model/label_encoders.pkl")
print("Saved model/scaler.pkl")
print("Saved model/label_encoders.pkl")

## 10. Apply SMOTE on training set ONLY
SMOTE is applied **only to X_train_scaled** - the test set is never touched.
This prevents data leakage and gives honest evaluation metrics.

In [ ]:
print(f"Before SMOTE: Legit={( y_train==0).sum():,}  Fraud={(y_train==1).sum():,}")

smote = SMOTE(k_neighbors=SMOTE_K, random_state=RANDOM_STATE, sampling_strategy='auto')
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print(f"After  SMOTE: Legit={(y_train_sm==0).sum():,}  Fraud={(y_train_sm==1).sum():,}")
print(f"Total training rows: {len(y_train_sm):,}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (c0, c1, title) in zip(axes, [
    ((y_train==0).sum(),    (y_train==1).sum(),    "Before SMOTE"),
    ((y_train_sm==0).sum(), (y_train_sm==1).sum(), "After SMOTE"),
]):
    bars = ax.bar(['Legit (0)','Fraud (1)'], [c0, c1],
                  color=['#4a90d9','#e74c3c'], width=0.5)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+max(c0,c1)*0.01,
                f"{int(bar.get_height()):,}", ha='center', fontsize=11)
    ax.set_title(title, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

plt.suptitle("SMOTE Class Balancing", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("model/smote_balance.png", dpi=120, bbox_inches='tight')
plt.show()

## 11. Train XGBoost (primary model)

In [ ]:
print("Training XGBoost...")

xgb_model = xgb.XGBClassifier(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    gamma            = 0.1,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    eval_metric      = 'aucpr',
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
)

xgb_model.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test_scaled, y_test)],
    verbose=50
)

y_pred_xgb = xgb_model.predict(X_test_scaled)
y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

print("\n" + "="*52)
print("XGBoost - Test Set Results")
print("="*52)
print(classification_report(y_test, y_pred_xgb, target_names=['Legit','Fraud'], digits=4))
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_xgb):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, y_prob_xgb):.4f}")
print(f"F1 Fraud : {f1_score(y_test, y_pred_xgb):.4f}")

## 12. Train LightGBM (comparison)

In [ ]:
print("Training LightGBM...")

lgb_model = lgb.LGBMClassifier(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_samples = 20,
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
    verbose           = -1,
)
lgb_model.fit(X_train_sm, y_train_sm)
y_pred_lgb = lgb_model.predict(X_test_scaled)
y_prob_lgb = lgb_model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_lgb, target_names=['Legit','Fraud'], digits=4))
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_lgb):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, y_prob_lgb):.4f}")

## 13. Train Random Forest (baseline)

In [ ]:
print("Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators = 200,
    max_depth    = 12,
    class_weight = 'balanced',
    random_state = RANDOM_STATE,
    n_jobs       = -1,
)
rf_model.fit(X_train_sm, y_train_sm)
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf  = rf_model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_rf, target_names=['Legit','Fraud'], digits=4))
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_rf):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, y_prob_rf):.4f}")

## 14. Compare all models visually

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models_res = {
    'XGBoost'      : (y_prob_xgb, y_pred_xgb),
    'LightGBM'     : (y_prob_lgb, y_pred_lgb),
    'RandomForest' : (y_prob_rf,  y_pred_rf),
}
colors = ['#e74c3c', '#3498db', '#2ecc71']

# PR curves
ax = axes[0]
for (name,(prob,_)), color in zip(models_res.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    ax.plot(rec, prec, color=color, label=f"{name} AP={ap:.3f}", linewidth=2)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve"); ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# Bar metrics
ax = axes[1]
metrics = ['F1 Fraud','ROC-AUC','PR-AUC']
all_scores = {
    'XGBoost'     :[f1_score(y_test,y_pred_xgb), roc_auc_score(y_test,y_prob_xgb), average_precision_score(y_test,y_prob_xgb)],
    'LightGBM'    :[f1_score(y_test,y_pred_lgb), roc_auc_score(y_test,y_prob_lgb), average_precision_score(y_test,y_prob_lgb)],
    'RandomForest':[f1_score(y_test,y_pred_rf),  roc_auc_score(y_test,y_prob_rf),  average_precision_score(y_test,y_prob_rf)],
}
x = np.arange(len(metrics)); w = 0.25
for i,(name,scores,color) in enumerate(zip(all_scores.keys(),all_scores.values(),colors)):
    ax.bar(x+i*w, scores, width=w, label=name, color=color, alpha=0.85)
ax.set_xticks(x+w); ax.set_xticklabels(metrics); ax.set_ylim(0,1.1)
ax.set_title("Metric Comparison"); ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# Confusion matrix
ax = axes[2]
ConfusionMatrixDisplay(confusion_matrix(y_test,y_pred_xgb),
                       display_labels=['Legit','Fraud']).plot(ax=ax, colorbar=False, cmap='Reds')
ax.set_title("XGBoost Confusion Matrix")

plt.suptitle("Model Comparison - TrackEasy Fraud Detection", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("model/model_comparison.png", dpi=120, bbox_inches='tight')
plt.show()

## 15. Feature importance (XGBoost)

In [ ]:
importances = xgb_model.feature_importances_
sorted_idx  = np.argsort(importances)
feat_names  = [FEATURE_COLS[i] for i in sorted_idx]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(feat_names, importances[sorted_idx], color='#e74c3c', alpha=0.8)
for bar, val in zip(bars, importances[sorted_idx]):
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
            f"{val:.4f}", va='center', fontsize=9)
ax.set_xlabel("Feature Importance (gain)")
ax.set_title("XGBoost Feature Importances", fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig("model/feature_importance.png", dpi=120, bbox_inches='tight')
plt.show()

rule_map = {
    'quantity'            : 'Rule 4 - quantity anomaly (+7)',
    'failed_payment_count': 'Rule 2 - payment failures (+4)',
    'transaction_hour'    : 'Rule 3 - speed check (+3)',
    'account_age_days'    : 'Rule 4 amplifier',
    'transaction_amount'  : 'Rule 4 amplifier',
    'addresses_match'     : 'Rule 5 support',
    'payment_method'      : 'Order model field',
    'device_type'         : 'UserSession field',
}
print("\nFeature -> fraudServer.js alignment (by importance):")
for feat in [FEATURE_COLS[i] for i in np.argsort(importances)[::-1]]:
    print(f"  {feat:<26} {rule_map.get(feat,'general signal')}")

## 16. Threshold tuning
fraudServer.js blocks at riskScore > 6/10 (60%). We find the threshold
that maximises F1 on the fraud class to match this intent.

In [ ]:
thresholds = np.arange(0.1, 0.91, 0.01)
f1_thresh  = [f1_score(y_test, (y_prob_xgb >= t).astype(int)) for t in thresholds]

best_thresh = float(thresholds[np.argmax(f1_thresh)])
print(f"Optimal threshold : {best_thresh:.2f}")
print(f"Best F1 (fraud)   : {max(f1_thresh):.4f}\n")

for t in sorted(set([0.3, 0.4, 0.5, round(best_thresh,2), 0.6])):
    preds = (y_prob_xgb >= t).astype(int)
    tp = ((preds==1) & (y_test==1)).sum()
    fp = ((preds==1) & (y_test==0)).sum()
    fn = ((preds==0) & (y_test==1)).sum()
    prec = tp / max(tp+fp, 1)
    rec  = tp / max(tp+fn, 1)
    print(f"  t={t:.2f}  F1={f1_score(y_test,preds):.4f}  Recall={rec:.3f}  Precision={prec:.3f}")

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(thresholds, f1_thresh, color='#e74c3c', linewidth=2)
ax.axvline(best_thresh, color='#2c3e50', linestyle='--', label=f"Best={best_thresh:.2f}")
ax.axvline(0.5, color='gray', linestyle=':', label="Default=0.50")
ax.set_xlabel("Decision Threshold"); ax.set_ylabel("F1 Score (Fraud)")
ax.set_title("Threshold Tuning"); ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig("model/threshold_curve.png", dpi=120, bbox_inches='tight')
plt.show()

BEST_THRESHOLD = best_thresh
print(f"\nUsing threshold = {BEST_THRESHOLD:.2f} in ml_server.py")

## 17. 5-fold cross-validation (SMOTE inside each fold)

In [ ]:
print("5-fold CV with SMOTE applied inside each fold...")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_f1s = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train), 1):
    Xtr, Xval = X_train_scaled[tr_idx], X_train_scaled[val_idx]
    ytr, yval = y_train[tr_idx], y_train[val_idx]

    k = max(1, min(SMOTE_K, int((ytr==1).sum()) - 1))
    sm_fold = SMOTE(k_neighbors=k, random_state=RANDOM_STATE)
    Xtr_sm, ytr_sm = sm_fold.fit_resample(Xtr, ytr)

    fm = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        eval_metric='aucpr', random_state=RANDOM_STATE, n_jobs=-1
    )
    fm.fit(Xtr_sm, ytr_sm, verbose=False)
    f = f1_score(yval, fm.predict(Xval))
    cv_f1s.append(f)
    print(f"  Fold {fold}: F1={f:.4f}  (fraud in val: {yval.sum()})")

print(f"\n  Mean F1 : {np.mean(cv_f1s):.4f}")
print(f"  Std  F1 : {np.std(cv_f1s):.4f}")
print("Low std = stable model")

## 18. Save all model artefacts

In [ ]:
joblib.dump(xgb_model, "model/fraud_model.pkl")
print("Saved model/fraud_model.pkl")
joblib.dump(lgb_model, "model/fraud_model_lgb.pkl")
print("Saved model/fraud_model_lgb.pkl")

model_meta = {
    "model_type"      : "XGBoostClassifier",
    "feature_columns" : FEATURE_COLS,
    "threshold"       : BEST_THRESHOLD,
    "numeric_cols"    : NUMERIC_COLS,
    "categorical_cols": CATEGORICAL_COLS,
    "numeric_indices" : numeric_indices,
    "smote_k"         : SMOTE_K,
    "test_f1_fraud"   : round(float(f1_score(y_test, y_pred_xgb)), 4),
    "test_roc_auc"    : round(float(roc_auc_score(y_test, y_prob_xgb)), 4),
    "test_pr_auc"     : round(float(average_precision_score(y_test, y_prob_xgb)), 4),
    "cv_mean_f1"      : round(float(np.mean(cv_f1s)), 4),
    "cv_std_f1"       : round(float(np.std(cv_f1s)), 4),
}

with open("model/model_meta.json", "w") as f:
    json.dump(model_meta, f, indent=2)
print("Saved model/model_meta.json")

print("\nAll model files:")
for fname in sorted(os.listdir("model")):
    size = os.path.getsize(f"model/{fname}")
    print(f"  model/{fname:<35} {size/1024:.1f} KB")

## 19. Generate ml_server.py (Flask inference server)
Writes the inference server that fraudServer.js calls on port 5003.
Run with: `python ml_server.py`

In [ ]:
ml_server_code = '#!/usr/bin/env python3\n"""\nTrackEasy ML Inference Server - port 5003\nCalled by fraudServer.js during evaluate-transaction\n\nStart : python ml_server.py\nHealth: curl http://localhost:5003/health\nTest  : curl -X POST http://localhost:5003/predict \\\n          -H "Content-Type: application/json" \\\n          -d "{\\"transaction_amount\\":5000,\\"payment_method\\":\\"Card\\",\\"quantity\\":15}"\n"""\nimport json, joblib, numpy as np, os\nfrom flask import Flask, request, jsonify\n\napp  = Flask(__name__)\nBASE = os.path.dirname(os.path.abspath(__file__))\n\nmodel          = joblib.load(os.path.join(BASE, "model/fraud_model.pkl"))\nscaler         = joblib.load(os.path.join(BASE, "model/scaler.pkl"))\nlabel_encoders = joblib.load(os.path.join(BASE, "model/label_encoders.pkl"))\n\nwith open(os.path.join(BASE, "model/model_meta.json")) as f:\n    meta = json.load(f)\n\nFEATURE_COLS    = meta["feature_columns"]\nTHRESHOLD       = meta["threshold"]\nNUMERIC_COLS    = meta["numeric_cols"]\nCATEGORICAL_COLS= meta["categorical_cols"]\nNUMERIC_INDICES = meta["numeric_indices"]\n\nprint(f"Model loaded | features={len(FEATURE_COLS)} | threshold={THRESHOLD}")\n\n\ndef build_feature_vector(data: dict) -> np.ndarray:\n    row = []\n    for col in FEATURE_COLS:\n        val = data.get(col, 0)\n        if col in label_encoders:\n            le = label_encoders[col]\n            s = str(val).lower().strip()\n            val = int(le.transform([s])[0]) if s in le.classes_ else 0\n        try:\n            row.append(float(val))\n        except (TypeError, ValueError):\n            row.append(0.0)\n    return np.array(row, dtype=float).reshape(1, -1)\n\n\n@app.route("/predict", methods=["POST"])\ndef predict():\n    try:\n        data = request.get_json(force=True)\n        X = build_feature_vector(data)\n        X_scaled = X.copy()\n        if NUMERIC_INDICES:\n            X_scaled[0, NUMERIC_INDICES] = scaler.transform(\n                X[0, NUMERIC_INDICES].reshape(1, -1))[0]\n        prob          = float(model.predict_proba(X_scaled)[0, 1])\n        is_fraud      = int(prob >= THRESHOLD)\n        ml_risk_score = round(prob * 10, 2)\n        return jsonify({\n            "fraud_probability": round(prob, 4),\n            "ml_risk_score"    : ml_risk_score,\n            "is_fraud"         : is_fraud,\n            "threshold_used"   : THRESHOLD,\n        })\n    except Exception as e:\n        return jsonify({"error": str(e), "fraud_probability": 0.0,\n                        "ml_risk_score": 0.0, "is_fraud": 0}), 200\n\n\n@app.route("/health", methods=["GET"])\ndef health():\n    return jsonify({"status":"ok","model":meta.get("model_type"),\n                    "features":FEATURE_COLS,"threshold":THRESHOLD,\n                    "test_f1":meta.get("test_f1_fraud")})\n\n\nif __name__ == "__main__":\n    app.run(host="0.0.0.0", port=5003, debug=False)\n'

with open("ml_server.py", "w") as f:
    f.write(ml_server_code)

print("Written: ml_server.py")
print()
print("To start:")
print("  pip install flask")
print("  python ml_server.py")
print("  -> listens on http://localhost:5003")

## 20. fraudServer.js patch
Paste this block into `fraudServer.js` inside the `evaluate-transaction` route,
right after the line: `riskScore = Math.min(10, riskScore);`

In [ ]:
patch = "// ─────────────────────────────────────────────────────────────\n// PASTE IN fraudServer.js AFTER: riskScore = Math.min(10, riskScore);\n// ─────────────────────────────────────────────────────────────\ntry {\n    const mlPayload = {\n        transaction_amount   : transactionDetails?.totalAmount || 0,\n        payment_method       : transactionDetails?.paymentMethod || 'unknown',\n        device_type          : req.body.deviceType || 'unknown',\n        account_age_days     : req.body.accountAgeDays || 365,\n        failed_payment_count : failedPayments.length,\n        transaction_hour     : new Date().getHours(),\n        quantity             : transactionDetails?.items?.length || 1,\n        addresses_match      : req.body.addressesMatch !== undefined ? req.body.addressesMatch : 1,\n        customer_age         : req.body.customerAge || 30,\n        product_category     : transactionDetails?.category || 'unknown',\n    };\n\n    const mlResponse = await fetch('http://localhost:5003/predict', {\n        method : 'POST',\n        headers: { 'Content-Type': 'application/json' },\n        body   : JSON.stringify(mlPayload)\n    });\n    const mlResult = await mlResponse.json();\n\n    // Blend: 60% rule-based + 40% ML\n    const blended = (riskScore * 0.6) + (mlResult.ml_risk_score * 0.4);\n    riskScore = Math.min(10, Math.round(blended * 10) / 10);\n\n    if (mlResult.is_fraud) {\n        violationReasons.push(\n            'ML model flagged (prob: ' + (mlResult.fraud_probability * 100).toFixed(1) + '%)'\n        );\n    }\n    console.log('[ML] prob=' + mlResult.fraud_probability + '  blended=' + riskScore);\n\n} catch (mlErr) {\n    console.warn('[ML] Inference server unreachable - using rule-only score:', mlErr.message);\n}\n// ─────────────────────────────────────────────────────────────"
print(patch)

## 21. Final summary

In [ ]:
print("=" * 60)
print("  TRAINING COMPLETE")
print("=" * 60)

if os.path.exists("model/model_meta.json"):
    with open("model/model_meta.json") as f:
        meta = json.load(f)
    for k, v in meta.items():
        print(f"  {k:<20}: {v}")

print()
print("Files generated:")
for fname in sorted(os.listdir("model")):
    print(f"  model/{fname}")
print("  ml_server.py")
print()
print("Integration steps:")
print("  1. Copy model/ and ml_server.py into TrackEasy/fraud-service/")
print("  2. pip install flask joblib xgboost scikit-learn lightgbm")
print("  3. python ml_server.py          (port 5003)")
print("  4. Paste Cell 20 snippet into fraudServer.js")
print("  5. node fraudServer.js          (port 5002)")
print("  6. Place a test order -> Manager Dashboard shows ML flag")